# Feature Extraction Pipeline
Run each cell in order. Only the **Configuration** cell needs editing.

In [ ]:
# ── Configuration ────────────────────────────────────────────

# 'A' = panoramic images  |  'B' = raw Ladybug images
MODE = 'B'

IMAGE_FOLDER = r'C:\Users\egkouvra\Downloads\images'

# Required for MODE = 'B'; leave empty for MODE = 'A'
CAL_FILE = r'D:\05_Other\GitClones\feature-extraction-from-ladybug-camera\data\Ladybug5_plus\ladybug20344317.cal'

MODELS = [
    {'model': 'COCO',         'model_type': 'OD'},
    {'model': 'Crosswalk',    'model_type': 'OD'},
    {'model': 'Traffic_Sign', 'model_type': 'OD'},
    # {'model': 'Safety_Cones', 'model_type': 'OD'},
    # {'model': 'COCO',         'model_type': 'P'},
    # {'model': 'Cityscapes',   'model_type': 'P'},
]

# Path to GET EOP CSV (tab-separated); leave empty to skip steps 3 and 4
EOP_CSV = r'D:\05_Other\GitClones\feature-extraction-from-ladybug-camera\data\GET\import_locations.csv'

# How to compute 3-D coordinates (requires EOP_CSV):
#   'association'  — multi-view grouping + forward intersection
#   'intersection' — classic N-ray WLS per named point
INTERSECTION_MODE = 'association'

In [ ]:
# ── Imports ───────────────────────────────────────────────────

import os, sys
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

# Resolve lib/ regardless of where the Jupyter kernel was launched from.
# __vsc_ipynb_file__ is injected by the VSCode notebook extension.
_nb_file = globals().get('__vsc_ipynb_file__') or os.path.abspath(__file__ if '__file__' in dir() else '')
_lib = os.path.normpath(os.path.join(os.path.dirname(_nb_file), '..', 'lib'))
if _lib not in sys.path:
    sys.path.insert(0, _lib)
print('lib path:', _lib)

from Detectron            import run_detection
from forward_intersection import run_intersection
from point_association    import associate
print('Imports OK')

In [ ]:
# ── Step 1 — Detection ────────────────────────────────────────
# Runs the object detection models on the images and writes:
#   output/coords/image_coords.csv   (panoramic pixel coordinates)
#   output/coords/raw_coords.csv     (raw pixel coordinates, MODE B only)

assert IMAGE_FOLDER, "Set IMAGE_FOLDER in the Configuration cell."
if MODE == 'B':
    assert CAL_FILE, "Set CAL_FILE in the Configuration cell for MODE B."

coords_csv = run_detection(
    IMAGE_FOLDER,
    MODELS,
    mode     = MODE,
    cal_file = CAL_FILE if MODE == 'B' else None,
)

print(f'\nDetection complete -> {coords_csv}')

In [ ]:
# ── (Optional) Set coords_csv manually to run Step 2 without Step 1 ─
# Leave empty to use the output of Step 1 above.
# Always point to image_coords.csv (panoramic coordinates), not raw_coords.csv.

_coords_override = r''

if _coords_override:
    coords_csv = _coords_override
print('coords_csv:', coords_csv)

In [ ]:
# ── Step 2 — 3-D Coordinates ──────────────────────────────────
# Requires EOP_CSV. Skipped if EOP_CSV is empty.
#
# association  -> groups detections of the same physical object across
#                 captures, then computes one EGSA87 point per group.
# intersection -> classic N-ray WLS, one point per named detection.

if not EOP_CSV:
    print('EOP_CSV not set — skipping 3-D coordinate computation.')
elif INTERSECTION_MODE == 'association':
    associate(coords_csv, EOP_CSV)
else:
    run_intersection(coords_csv, EOP_CSV)